In [1]:
!pip install mlflow scikit-learn pandas flask


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!wget --no-verbose https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv -O data/WA_Fn-UseC_-Telco-Customer-Churn.csv

2026-10-22 18:24:19 URL:https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv [970457/970457] -> "data/WA_Fn-UseC_-Telco-Customer-Churn.csv" [1]


In [4]:
# Load and Explore Data
import pandas as pd

# Load data
data = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Display the first few rows
print(data.head())

# Check for missing values
print(data.isnull().sum())

# Get basic info about the data
print(data.info())

# Describe the data
print(data.describe())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [8]:
!python train.py

/opt/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Model accuracy: 0.7473385379701917
Model precision: 0.5142378559463987
Model recall: 0.8230563002680965
Model f1-score: 0.6329896907216495


In [7]:
!mlflow ui

[2026-10-22 18:38:35 +0000] [41715] [INFO] Starting gunicorn 21.2.0
[2026-10-22 18:38:35 +0000] [41715] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2026-10-22 18:38:35 +0000] [41715] [ERROR] Retrying in 1 second.
[2026-10-22 18:38:36 +0000] [41715] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2026-10-22 18:38:36 +0000] [41715] [ERROR] Retrying in 1 second.
[2026-10-22 18:38:37 +0000] [41715] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2026-10-22 18:38:37 +0000] [41715] [ERROR] Retrying in 1 second.
[2026-10-22 18:38:38 +0000] [41715] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2026-10-22 18:38:38 +0000] [41715] [ERROR] Retrying in 1 second.
[2026-10-22 18:38:39 +0000] [41715] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2026-10-22 18:38:39 +0000] [41715] [ERROR] Retrying in 1 second.
[2026-10-22 18:38:40 +0000] [41715] [ERROR] Can't connect to ('127.0.0.1', 5000)
Running the mlflow server failed. Please see the logs above for details.


In [9]:
    import os

    # Set the MLFLOW_TRACKING_URI environment variable
    os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

    # Run the serve.py script in the background using nohup
    !nohup python serve.py > server.log 2>&1 &

OSError: Background processes not supported.

In [10]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

script = Path("serve.py").resolve()
if not script.is_file():
    raise FileNotFoundError(f"Can't find serve.py from {Path.cwd()}")

with open(script.with_name("server.log"), "a") as log:
    serve_process = subprocess.Popen(
        [sys.executable, str(script)],
        cwd=script.parent,
        env=os.environ.copy(),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print(f"Started serve.py (PID {serve_process.pid}). Check server.log for output.")

Started serve.py (PID 42466). Check server.log for output.


In [11]:
curl -X POST -H "Content-Type: application/json" -d '[{"SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No", "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service", "OnlineSecurity": "No", "OnlineBackup": "Yes", "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No", "StreamingMovies": "No", "PaperlessBilling": "Yes", "MonthlyCharges": 29.85, "TotalCharges": 29.85, "InternetService_DSL": 1, "InternetService_Fiber optic": 0, "InternetService_No": 0, "Contract_Month-to-month": 1, "Contract_One year": 0, "Contract_Two year": 0, "PaymentMethod_Bank transfer (automatic)": 0, "PaymentMethod_Credit card (automatic)": 0, "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0}]' http://localhost:5000/predict

SyntaxError: invalid syntax (2009670201.py, line 1)

In [12]:
curl -X POST -H "Content-Type: application/json" -d '[{"SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No", "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service", "OnlineSecurity": "No", "OnlineBackup": "Yes", "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No", "StreamingMovies": "No", "PaperlessBilling": "Yes", "MonthlyCharges": 29.85, "TotalCharges": 29.85, "InternetService_DSL": 1, "InternetService_Fiber optic": 0, "InternetService_No": 0, "Contract_Month-to-month": 1, "Contract_One year": 0, "Contract_Two year": 0, "PaymentMethod_Bank transfer (automatic)": 0, "PaymentMethod_Credit card (automatic)": 0, "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0}]' http://localhost:5000/predict

SyntaxError: invalid syntax (2009670201.py, line 1)

In [13]:
!curl -X POST -H "Content-Type: application/json" \
  -d '[{"SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No", "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service", "OnlineSecurity": "No", "OnlineBackup": "Yes", "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No", "StreamingMovies": "No", "PaperlessBilling": "Yes", "MonthlyCharges": 29.85, "TotalCharges": 29.85, "InternetService_DSL": 1, "InternetService_Fiber optic": 0, "InternetService_No": 0, "Contract_Month-to-month": 1, "Contract_One year": 0, "Contract_Two year": 0, "PaymentMethod_Bank transfer (automatic)": 0, "PaymentMethod_Credit card (automatic)": 0, "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0}]' \
  http://localhost:5000/predict

<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>


In [14]:
print("Process status:", serve_process.poll())  # None means it is still running
print(Path("server.log").read_text()[-3000:])

Process status: 1

Traceback (most recent call last):
  File "/app/serve.py", line 31, in <module>
    model = get_latest_model()
            ^^^^^^^^^^^^^^^^^^
  File "/app/serve.py", line 29, in get_latest_model
    return mlflow.pyfunc.load_model(model_uri)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/pyfunc/__init__.py", line 634, in load_model
    local_path = _download_artifact_from_uri(artifact_uri=model_uri, output_path=dst_path)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/tracking/artifact_utils.py", line 100, in _download_artifact_from_uri
    return get_artifact_repository(artifact_uri=root_uri).download_artifacts(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/store/artifact/runs_artifact_repo.py", line 125, in download_artifacts
    re

In [15]:
import requests

response = requests.post(
    "http://localhost:5000/predict",
    json=[your_input_dictionary],
    timeout=10,
)
print("HTTP status:", response.status_code)
print("Response:", response.text)


NameError: name 'your_input_dictionary' is not defined

In [17]:
import requests

input_row = {
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "No",
    "MultipleLines": "No phone service",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "PaperlessBilling": "Yes",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85,
    "InternetService_DSL": 1,
    "InternetService_Fiber optic": 0,
    "InternetService_No": 0,
    "Contract_Month-to-month": 1,
    "Contract_One year": 0,
    "Contract_Two year": 0,
    "PaymentMethod_Bank transfer (automatic)": 0,
    "PaymentMethod_Credit card (automatic)": 0,
    "PaymentMethod_Electronic check": 1,
    "PaymentMethod_Mailed check": 0,
}

response = requests.post(
    "http://localhost:5000/experiments",
    json=[input_row],
    timeout=10,
)
print("HTTP status:", response.status_code)
print("Response:", response.text)

HTTP status: 404
Response: <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



In [18]:
input_row = {
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "No",
    "MultipleLines": "No phone service",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "PaperlessBilling": "Yes",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85,
    "InternetService_DSL": 1,
    "InternetService_Fiber optic": 0,
    "InternetService_No": 0,
    "Contract_Month-to-month": 1,
    "Contract_One year": 0,
    "Contract_Two year": 0,
    "PaymentMethod_Bank transfer (automatic)": 0,
    "PaymentMethod_Credit card (automatic)": 0,
    "PaymentMethod_Electronic check": 1,
    "PaymentMethod_Mailed check": 0,
}

response = requests.post(
    "http://localhost:5000/predict",
    json=[input_row],
    timeout=10,
)
print(response.status_code, response.text)

404 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



In [19]:
import os
import sys
import subprocess
from pathlib import Path

script = Path(
    "/home/du_357155-1791492091/deloitte-ai-solution-architect-v2.0-q1-2026-main (3)/deloitte-ai-solution-architect-v2.0-q1-2026-main/modules/sa-docker/serve.py"
)

if not script.is_file():
    raise FileNotFoundError(script)

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

log_file = open(script.with_name("server.log"), "a")
serve_process = subprocess.Popen(
    [sys.executable, str(script)],
    cwd=script.parent,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Started serve.py; PID:", serve_process.pid)

FileNotFoundError: /home/du_357155-1791492091/deloitte-ai-solution-architect-v2.0-q1-2026-main (3)/deloitte-ai-solution-architect-v2.0-q1-2026-main/modules/sa-docker/serve.py

In [20]:
from pathlib import Path

print("Process status:", serve_process.poll())  # None means it is still running
print(Path("server.log").read_text()[-3000:])

Process status: 1

Traceback (most recent call last):
  File "/app/serve.py", line 31, in <module>
    model = get_latest_model()
            ^^^^^^^^^^^^^^^^^^
  File "/app/serve.py", line 29, in get_latest_model
    return mlflow.pyfunc.load_model(model_uri)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/pyfunc/__init__.py", line 634, in load_model
    local_path = _download_artifact_from_uri(artifact_uri=model_uri, output_path=dst_path)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/tracking/artifact_utils.py", line 100, in _download_artifact_from_uri
    return get_artifact_repository(artifact_uri=root_uri).download_artifacts(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/store/artifact/runs_artifact_repo.py", line 125, in download_artifacts
    re

In [21]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
run = mlflow.get_run("e66cd0985a594a728e9fa0725ef2f872")
print(run.info.artifact_uri)

mlflow-artifacts:/0/e66cd0985a594a728e9fa0725ef2f872/artifacts


In [22]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
print(client.list_artifacts("e66cd0985a594a728e9fa0725ef2f872"))

[<FileInfo: file_size=None, is_dir=True, path='iris-random-forest-model'>]


In [23]:
    import os

    # Set the MLFLOW_TRACKING_URI environment variable
    os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

    # Run the serve.py script in the background using nohup
    !nohup python serve.py > server.log 2>&1 &

OSError: Background processes not supported.

In [24]:
import os
import sys
import subprocess
from pathlib import Path

script = Path("serve.py").resolve()
if not script.is_file():
    raise FileNotFoundError(f"serve.py not found at {script}")

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

with open(script.with_name("server.log"), "a") as log:
    serve_process = subprocess.Popen(
        [sys.executable, str(script)],
        cwd=script.parent,
        env=os.environ.copy(),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Started with PID:", serve_process.pid)

Started with PID: 43564


In [25]:
print("Process status:", serve_process.poll())  # None means running; 1 means it exited
print(Path("server.log").read_text()[-3000:])

Process status: 1
^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/store/artifact/runs_artifact_repo.py", line 125, in download_artifacts
    return self.repo.download_artifacts(artifact_path, dst_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/mlflow/store/artifact/artifact_repo.py", line 222, in download_artifacts
    raise MlflowException(
mlflow.exceptions.MlflowException: The following failures occurred while downloading one or more artifacts from http://localhost:5000/api/2.0/mlflow-artifacts/artifacts/0/e66cd0985a594a728e9fa0725ef2f872/artifacts:
##### File model #####
API request to http://localhost:5000/api/2.0/mlflow-artifacts/artifacts/0/e66cd0985a594a728e9fa0725ef2f872/artifacts/model failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/0/e66cd0985a594a728e9fa0725ef2f872/artifacts/model (Caused by ResponseErro